# Agent Memory: Long-Term — Persistent Memory with Vector Stores

## What This Notebook Teaches You

Short-term memory (Notebook 10) manages context within a single conversation. But what about agents that run across **multiple sessions**? A customer support agent should remember that a user had a billing issue last week. A coding assistant should recall that a project uses PostgreSQL, not MySQL.

This notebook builds **long-term memory (LTM)** — memory that persists beyond a single conversation, backed by vector stores.

By the end, you will:

1. **Understand LTM theory** — episodic vs. semantic memory, drawing from cognitive science
2. **Build EpisodicMemory** — FAISS-backed store for timestamped experiences
3. **Build SemanticMemory** — deduplicated store for facts and knowledge
4. **Combine into LongTermMemory** — unified class with recall, consolidation, and decay
5. **Build a memory-enhanced agent** — ReAct agent that recalls relevant memories each step
6. **Run a cross-session demo** — agent learns in sessions 1 & 2, recalls in session 3

---

> **Prerequisites**: Notebooks 01–10 (especially 10 for short-term memory context).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~50–60 minutes.

## Part 0: Environment Setup

We need the LLM (Qwen3-8B) for reasoning plus an **embedding model** (BGE-base) and **FAISS** for vector similarity search.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import torch
import time
import json
import re
import math
import numpy as np
import faiss
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# Mount Google Drive for model caching
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

# Load embedding model for memory/retrieval
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def embed_texts(texts):
    """Embed a list of texts using BGE model. Returns numpy array."""
    return embed_model.encode(texts, normalize_embeddings=True)

print(f"✓ LLM loaded: {MODEL_NAME}")
print(f"  GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"✓ Embedding model loaded: BAAI/bge-base-en-v1.5 (768-dim)")

## 1. Beyond the Context Window

### The Persistence Problem

Short-term memory strategies (Notebook 10) all share a fundamental limitation: **memory dies when the session ends**. This is like having a brilliant colleague with complete amnesia — every day you meet them for the first time.

### Human Memory as Architecture Guide

Cognitive science distinguishes several memory types:

| Memory Type | Duration | Example | Agent Analog |
|------------|----------|---------|-------------|
| **Working Memory** | Seconds | Mental arithmetic | Context window |
| **Short-Term** | Minutes–hours | Conversation context | Sliding window / summary |
| **Episodic** | Days–years | "Last Tuesday I debugged a CORS issue" | Timestamped vector store |
| **Semantic** | Permanent | "Python lists are mutable" | Deduplicated fact store |
| **Procedural** | Permanent | How to ride a bike | Fine-tuned model weights |

This notebook implements **episodic** and **semantic** long-term memory.

### Research Foundation

Park et al. (2023) — *"Generative Agents: Interactive Simulacra of Human Behavior"* introduced persistent memory for LLM agents. Their 25 agents maintained episodic memories, reflected on them, and formed plans — producing remarkably human-like behavior in a simulated town. Our implementation draws directly from their architecture.

In [ ]:
# Utility: cosine similarity for deduplication and comparison
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

# Demonstrate embedding model
test_texts = [
    "The API uses JWT tokens for authentication",
    "JSON Web Tokens handle user auth in the REST API",
    "The database runs on PostgreSQL 15",
    "Python lists are mutable ordered collections",
]

embeddings = embed_texts(test_texts)
print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

print("\nSemantic similarity matrix:")
print(f"{'':>50}", end="")
for i in range(len(test_texts)):
    print(f" [{i}]", end="  ")
print()

for i, text_i in enumerate(test_texts):
    short = text_i[:48] + ".." if len(text_i) > 50 else text_i
    print(f"[{i}] {short:>48}", end=" ")
    for j in range(len(test_texts)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"{sim:.2f}", end="  ")
    print()

print("\n✓ Texts [0] and [1] are semantically similar (same concept, different words)")
print("  Texts [0] and [2] are different topics (auth vs. database)")

## 2. Episodic Memory — Timestamped Experiences

Episodic memory stores **what happened when** — concrete experiences with temporal context. Each memory has:

- **Content**: What happened (text description)
- **Timestamp**: When it happened
- **Embedding**: Vector representation for similarity search
- **Metadata**: Session ID, importance score, access count

Retrieval is by **semantic similarity** — "What do I remember about authentication?" finds auth-related episodes regardless of exact wording.

In [ ]:
class EpisodicMemory:
    """Vector-store backed memory for timestamped experiences."""

    def __init__(self, embedding_dim=768):
        self.embedding_dim = embedding_dim
        self.index = faiss.IndexFlatIP(embedding_dim)  # inner product (cosine with normalized vecs)
        self.memories = []  # parallel list of memory metadata

    def store(self, content, session_id="default", importance=5.0, metadata=None):
        """Store an episodic memory."""
        embedding = embed_texts([content])[0]  # normalized by embed_texts
        self.index.add(np.array([embedding], dtype=np.float32))

        memory_entry = {
            "content": content,
            "timestamp": time.time(),
            "session_id": session_id,
            "importance": importance,
            "access_count": 0,
            "metadata": metadata or {},
            "index": len(self.memories),
        }
        self.memories.append(memory_entry)
        return memory_entry

    def recall(self, query, k=5, min_similarity=0.3):
        """Recall memories by semantic similarity to a query."""
        if self.index.ntotal == 0:
            return []

        query_embedding = embed_texts([query])[0]
        k = min(k, self.index.ntotal)
        scores, indices = self.index.search(np.array([query_embedding], dtype=np.float32), k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0 or score < min_similarity:
                continue
            memory = self.memories[idx].copy()
            memory["similarity"] = float(score)
            memory["access_count"] = self.memories[idx]["access_count"] + 1
            self.memories[idx]["access_count"] += 1
            results.append(memory)

        return results

    def recall_by_session(self, session_id):
        """Get all memories from a specific session."""
        return [m for m in self.memories if m["session_id"] == session_id]

    def size(self):
        return len(self.memories)

    def __repr__(self):
        return f"EpisodicMemory({self.size()} episodes)"


# Build episodic memory from a simulated debugging session
episodic = EpisodicMemory()

# Session 1: User debugging an API issue
session1_events = [
    ("User reported 500 errors on the /api/users endpoint when processing large payloads", 7.0),
    ("Investigated server logs: OutOfMemoryError in the request handler for POST /api/users", 8.0),
    ("Root cause: unbounded JSON body parsing with no size limit configured", 9.0),
    ("Fix applied: added max_content_length=10MB to the Flask config", 9.0),
    ("Deployed fix to staging. All tests passing. 500 errors resolved.", 8.0),
]

for content, importance in session1_events:
    episodic.store(content, session_id="session-001", importance=importance)

# Session 2: Different issue
session2_events = [
    ("User asked about setting up CI/CD pipeline for the Python project", 5.0),
    ("Recommended GitHub Actions with pytest, flake8, and Docker build steps", 6.0),
    ("Created .github/workflows/ci.yml with test and deploy stages", 8.0),
    ("Added Dockerfile with multi-stage build for production deployment", 7.0),
    ("Pipeline running successfully: tests pass in 2 minutes, deploy in 5 minutes", 7.0),
]

for content, importance in session2_events:
    episodic.store(content, session_id="session-002", importance=importance)

print(f"Stored {episodic.size()} episodic memories across 2 sessions")
print(f"\n{'='*60}")

# Test retrieval
queries = [
    "What was the API error we fixed?",
    "How did we set up the CI/CD pipeline?",
    "What Docker configuration did we use?",
    "Was there a memory issue?",
]

for query in queries:
    print(f"\n🔍 Query: '{query}'")
    results = episodic.recall(query, k=2)
    for r in results:
        print(f"   [{r['similarity']:.3f}] (session {r['session_id']}): {r['content'][:80]}")

## 3. Semantic Memory — Deduplicated Fact Store

While episodic memory stores **events**, semantic memory stores **facts** — general knowledge the agent has acquired. Key difference: semantic memory **deduplicates** — if the agent learns the same fact twice, it updates rather than duplicates.

Deduplication uses cosine similarity: if a new fact is >threshold similar to an existing one, we update instead of insert.

In [ ]:
class SemanticMemory:
    """Deduplicated fact store with vector similarity search."""

    def __init__(self, embedding_dim=768, dedup_threshold=0.85):
        self.embedding_dim = embedding_dim
        self.dedup_threshold = dedup_threshold
        self.index = faiss.IndexFlatIP(embedding_dim)
        self.facts = []
        self.embeddings_list = []  # keep embeddings for dedup comparison

    def store_fact(self, fact, category="general", source="agent"):
        """Store a fact, deduplicating against existing facts."""
        embedding = embed_texts([fact])[0]

        # Check for duplicates
        if self.index.ntotal > 0:
            scores, indices = self.index.search(
                np.array([embedding], dtype=np.float32), min(5, self.index.ntotal)
            )
            for score, idx in zip(scores[0], indices[0]):
                if idx >= 0 and score >= self.dedup_threshold:
                    # Update existing fact
                    old_fact = self.facts[idx]["content"]
                    self.facts[idx]["content"] = fact
                    self.facts[idx]["updated_at"] = time.time()
                    self.facts[idx]["update_count"] += 1
                    # Update embedding
                    self.embeddings_list[idx] = embedding
                    self._rebuild_index()
                    return {"action": "updated", "old": old_fact, "new": fact, "similarity": float(score)}

        # New fact — add it
        self.index.add(np.array([embedding], dtype=np.float32))
        self.embeddings_list.append(embedding)
        fact_entry = {
            "content": fact,
            "category": category,
            "source": source,
            "created_at": time.time(),
            "updated_at": time.time(),
            "update_count": 0,
            "access_count": 0,
        }
        self.facts.append(fact_entry)
        return {"action": "added", "fact": fact}

    def _rebuild_index(self):
        """Rebuild FAISS index from current embeddings."""
        self.index = faiss.IndexFlatIP(self.embedding_dim)
        if self.embeddings_list:
            self.index.add(np.array(self.embeddings_list, dtype=np.float32))

    def query(self, question, k=5, min_similarity=0.3):
        """Find facts relevant to a question."""
        if self.index.ntotal == 0:
            return []

        query_embedding = embed_texts([question])[0]
        k = min(k, self.index.ntotal)
        scores, indices = self.index.search(np.array([query_embedding], dtype=np.float32), k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0 or score < min_similarity:
                continue
            fact = self.facts[idx].copy()
            fact["similarity"] = float(score)
            self.facts[idx]["access_count"] += 1
            results.append(fact)

        return results

    def all_facts(self):
        return [f["content"] for f in self.facts]

    def size(self):
        return len(self.facts)

    def __repr__(self):
        return f"SemanticMemory({self.size()} facts, dedup_threshold={self.dedup_threshold})"


# Demonstrate semantic memory with deduplication
semantic = SemanticMemory(dedup_threshold=0.85)

# Store facts — some are duplicates of each other
facts_to_store = [
    ("The project uses Python 3.11 with FastAPI framework", "tech_stack"),
    ("Database is PostgreSQL 15 with async SQLAlchemy", "tech_stack"),
    ("Authentication uses JWT tokens with 15-minute expiry", "auth"),
    ("The project uses Python 3.11 and FastAPI for the backend", "tech_stack"),   # near-duplicate of #1
    ("JWT authentication tokens expire after 15 minutes", "auth"),                 # near-duplicate of #3
    ("Redis is used for caching with a 5-minute TTL", "infrastructure"),
    ("The team uses 2-week sprints with Monday standups", "process"),
    ("Deployment target is AWS ECS with Fargate", "infrastructure"),
    ("The Python project runs on version 3.11 with FastAPI", "tech_stack"),        # near-duplicate of #1
]

print("Semantic Memory — Deduplication Demo")
print("=" * 70)

for fact, category in facts_to_store:
    result = semantic.store_fact(fact, category=category)
    action = result["action"]
    if action == "updated":
        print(f"  🔄 UPDATED (sim={result['similarity']:.3f}): {fact[:60]}")
        print(f"       was: {result['old'][:60]}")
    else:
        print(f"  ✅ ADDED:   {fact[:60]}")

print(f"\n📊 Stored {len(facts_to_store)} facts → {semantic.size()} unique facts (deduplication removed {len(facts_to_store) - semantic.size()})")

# Query semantic memory
print(f"\n{'='*70}")
print("Fact retrieval:")
for q in ["What programming language does the project use?", "How is caching configured?", "What is the sprint schedule?"]:
    results = semantic.query(q, k=2)
    print(f"\n  🔍 '{q}'")
    for r in results:
        print(f"     [{r['similarity']:.3f}] {r['content'][:70]}")

## 4. LongTermMemory — Unified Interface

Now we combine episodic and semantic memory into a single **LongTermMemory** class with:

- `store_episode(text)` — store a timestamped experience
- `store_fact(text)` — store a deduplicated fact
- `recall(query, k=5)` — search across both memory types
- `consolidate()` — extract facts from episodes (like sleeping on it)

In [ ]:
class LongTermMemory:
    """Unified long-term memory combining episodic and semantic stores."""

    def __init__(self, embedding_dim=768, dedup_threshold=0.85):
        self.episodic = EpisodicMemory(embedding_dim)
        self.semantic = SemanticMemory(embedding_dim, dedup_threshold)

    def store_episode(self, content, session_id="default", importance=5.0, metadata=None):
        """Store a timestamped experience."""
        return self.episodic.store(content, session_id, importance, metadata)

    def store_fact(self, fact, category="general", source="agent"):
        """Store a deduplicated fact."""
        return self.semantic.store_fact(fact, category, source)

    def recall(self, query, k=5, min_similarity=0.3):
        """Search across both episodic and semantic memory.

        Returns results from both stores, sorted by similarity.
        """
        episodic_results = self.episodic.recall(query, k=k, min_similarity=min_similarity)
        for r in episodic_results:
            r["memory_type"] = "episodic"

        semantic_results = self.semantic.query(query, k=k, min_similarity=min_similarity)
        for r in semantic_results:
            r["memory_type"] = "semantic"

        # Merge and sort by similarity
        all_results = episodic_results + semantic_results
        all_results.sort(key=lambda x: x["similarity"], reverse=True)

        return all_results[:k]

    def consolidate(self):
        """Extract facts from recent episodes using the LLM.

        This is like 'sleeping on it' — converting experiences into knowledge.
        """
        # Get recent episodes that haven't been consolidated
        recent = self.episodic.memories[-10:]  # last 10 episodes
        if not recent:
            return []

        episodes_text = "\n".join([f"- {m['content']}" for m in recent])

        extraction_prompt = [
            {"role": "system", "content": "Extract general facts and knowledge from these agent experiences. Output one fact per line. Only output facts, no commentary. Be concise."},
            {"role": "user", "content": f"Experiences:\n{episodes_text}"}
        ]

        response = generate(extraction_prompt, max_new_tokens=300, temperature=0.3)

        extracted_facts = []
        for line in response.strip().split("\n"):
            line = line.strip().lstrip("- •0123456789.")
            if len(line) > 10:  # skip empty/trivial lines
                result = self.store_fact(line, category="consolidated", source="consolidation")
                extracted_facts.append((line, result["action"]))

        return extracted_facts

    def format_for_context(self, query, k=5):
        """Format recalled memories for inclusion in an LLM prompt."""
        results = self.recall(query, k=k)
        if not results:
            return "No relevant memories found."

        lines = ["Relevant memories:"]
        for r in results:
            mem_type = r["memory_type"]
            content = r.get("content", r.get("content", ""))
            sim = r["similarity"]
            lines.append(f"  [{mem_type}, sim={sim:.2f}] {content}")

        return "\n".join(lines)

    def stats(self):
        return {
            "episodic_count": self.episodic.size(),
            "semantic_count": self.semantic.size(),
            "total": self.episodic.size() + self.semantic.size(),
        }

    def __repr__(self):
        s = self.stats()
        return f"LongTermMemory(episodes={s['episodic_count']}, facts={s['semantic_count']})"


# Build LTM from our earlier examples
ltm = LongTermMemory()

# Store episodes
for content, importance in session1_events:
    ltm.store_episode(content, session_id="session-001", importance=importance)
for content, importance in session2_events:
    ltm.store_episode(content, session_id="session-002", importance=importance)

# Store some direct facts
direct_facts = [
    ("The API is built with Flask running on port 5000", "tech_stack"),
    ("Production uses 3 replicas behind an nginx load balancer", "infrastructure"),
    ("Database backups run daily at 2 AM UTC", "operations"),
]
for fact, cat in direct_facts:
    ltm.store_fact(fact, category=cat)

print(ltm)
print()

# Test unified recall
test_queries = [
    "What errors did we encounter?",
    "How is the CI/CD pipeline configured?",
    "What infrastructure do we use?",
]

for query in test_queries:
    print(f"\n🔍 '{query}'")
    results = ltm.recall(query, k=3)
    for r in results:
        print(f"   [{r['memory_type']:>9}, {r['similarity']:.3f}] {r.get('content', '')[:70]}")

In [ ]:
# Test consolidation — extract facts from episodes
print("Memory Consolidation — Extracting Facts from Episodes")
print("=" * 60)
print(f"Before: {ltm.stats()}")

extracted = ltm.consolidate()

print(f"\nExtracted {len(extracted)} facts from episodes:")
for fact, action in extracted:
    print(f"  {'🔄' if action == 'updated' else '✅'} [{action}] {fact[:70]}")

print(f"\nAfter: {ltm.stats()}")

## 5. Memory-Enhanced Agent

Now we integrate long-term memory into a ReAct agent. At each step, the agent:

1. **Recalls** relevant memories based on the current query
2. **Includes** recalled memories in its context
3. **Stores** important findings as new memories
4. **Persists** memories across sessions

In [ ]:
class MemoryEnhancedAgent:
    """ReAct agent with long-term memory integration."""

    SYSTEM_PROMPT = """You are a helpful assistant with long-term memory. You remember facts from previous conversations.

When you discover important facts or decisions, say: STORE_FACT: <fact>
When you have a significant experience worth remembering, say: STORE_EPISODE: <episode>

Answer based on both your knowledge and your recalled memories.
Always acknowledge when you're using information from memory."""

    def __init__(self, ltm=None):
        self.ltm = ltm or LongTermMemory()
        self.session_id = f"session-{int(time.time())}"
        self.conversation = []  # current session messages

    def step(self, user_input, verbose=True):
        """Process one user message with memory augmentation."""

        # 1. Recall relevant memories
        memories_text = self.ltm.format_for_context(user_input, k=5)

        # 2. Build prompt with memories
        system_content = self.SYSTEM_PROMPT
        if memories_text != "No relevant memories found.":
            system_content += f"\n\n{memories_text}"

        messages = [{"role": "system", "content": system_content}]
        messages.extend(self.conversation[-10:])  # recent conversation
        messages.append({"role": "user", "content": user_input})

        # 3. Generate response
        response = generate(messages, max_new_tokens=400, temperature=0.7)

        # 4. Store conversation
        self.conversation.append({"role": "user", "content": user_input})
        self.conversation.append({"role": "assistant", "content": response})

        # 5. Store episode
        self.ltm.store_episode(
            f"User asked: {user_input[:100]}. Agent responded about: {response[:100]}",
            session_id=self.session_id, importance=5.0
        )

        # 6. Extract and store any explicit STORE commands
        for line in response.split("\n"):
            line = line.strip()
            if line.startswith("STORE_FACT:"):
                fact = line[11:].strip()
                if fact:
                    self.ltm.store_fact(fact, source="agent-explicit")
                    if verbose:
                        print(f"   💾 Stored fact: {fact[:60]}")
            elif line.startswith("STORE_EPISODE:"):
                episode = line[14:].strip()
                if episode:
                    self.ltm.store_episode(episode, session_id=self.session_id, importance=7.0)
                    if verbose:
                        print(f"   💾 Stored episode: {episode[:60]}")

        if verbose:
            recalled = self.ltm.recall(user_input, k=3)
            if recalled:
                mem_types = [r["memory_type"] for r in recalled]
                print(f"   🧠 Recalled {len(recalled)} memories: {mem_types}")

        return response

    def new_session(self):
        """Start a new session (simulates closing and reopening the agent)."""
        old_session = self.session_id
        self.session_id = f"session-{int(time.time())}"
        self.conversation = []  # clear conversation but LTM persists
        return old_session

# Create agent with fresh LTM
agent_ltm = LongTermMemory()
agent = MemoryEnhancedAgent(ltm=agent_ltm)

print("Memory-Enhanced Agent — Session 1")
print("=" * 60)

s1_inputs = [
    "I'm working on a Python web scraper that collects product prices from e-commerce sites.",
    "The scraper uses BeautifulSoup for parsing and stores results in a SQLite database.",
    "I'm having an issue: the scraper crashes when a product page has no price element.",
]

for user_input in s1_inputs:
    print(f"\n👤 {user_input}")
    response = agent.step(user_input)
    print(f"🤖 {response[:300]}")

print(f"\n📊 LTM after session 1: {agent.ltm.stats()}")

## 6. Cross-Session Demo

The critical test: can the agent remember information across sessions? We simulate three separate sessions where the agent accumulates knowledge.

In [ ]:
# Session 2 — different topic, same agent memory
old_session = agent.new_session()
print(f"\n{'='*60}")
print(f"Session 2 (previous: {old_session})")
print("=" * 60)

s2_inputs = [
    "Now I'm adding a notification feature. When a tracked price drops below a threshold, send an email.",
    "I'm using smtplib with Gmail SMTP. The threshold is stored per-product in the SQLite database.",
    "Got it working! Price alerts now send within 30 seconds of detection.",
]

for user_input in s2_inputs:
    print(f"\n👤 {user_input}")
    response = agent.step(user_input)
    print(f"🤖 {response[:300]}")

print(f"\n📊 LTM after session 2: {agent.ltm.stats()}")

In [ ]:
# Session 3 — test recall from previous sessions
old_session = agent.new_session()
print(f"\n{'='*60}")
print(f"Session 3 — Testing Cross-Session Recall (previous: {old_session})")
print("=" * 60)

s3_inputs = [
    "What project have we been working on?",
    "What database am I using for this project?",
    "Did we set up any notification features?",
    "What was the bug we encountered?",
]

for user_input in s3_inputs:
    print(f"\n👤 {user_input}")
    response = agent.step(user_input)
    print(f"🤖 {response[:300]}")

print(f"\n📊 LTM after session 3: {agent.ltm.stats()}")

## 7. Memory Importance and Decay

Not all memories should persist forever. Humans forget, and agents should too. We implement:

1. **Importance scoring** — higher-importance memories are retained longer
2. **Recency weighting** — recent memories are preferred over old ones
3. **Access-based reinforcement** — frequently accessed memories are strengthened
4. **Consolidation** — periodically extract lasting facts from episodes

In [ ]:
class DecayingMemory:
    """Memory system with importance scoring and temporal decay."""

    def __init__(self, embedding_dim=768, decay_rate=0.01):
        self.embedding_dim = embedding_dim
        self.decay_rate = decay_rate
        self.memories = []
        self.embeddings_list = []
        self.index = faiss.IndexFlatIP(embedding_dim)

    def store(self, content, importance=5.0, metadata=None):
        """Store a memory with importance score."""
        embedding = embed_texts([content])[0]
        self.index.add(np.array([embedding], dtype=np.float32))
        self.embeddings_list.append(embedding)

        self.memories.append({
            "content": content,
            "importance": importance,
            "created_at": time.time(),
            "last_accessed": time.time(),
            "access_count": 0,
            "metadata": metadata or {},
        })

    def compute_relevance(self, memory, similarity, current_time=None):
        """Compute final relevance combining similarity, importance, recency, and access."""
        current_time = current_time or time.time()
        age_hours = (current_time - memory["created_at"]) / 3600
        time_since_access = (current_time - memory["last_accessed"]) / 3600

        # Components
        semantic_score = similarity                             # [0, 1]
        importance_score = memory["importance"] / 10.0          # [0, 1]
        recency_score = math.exp(-self.decay_rate * age_hours)  # exponential decay
        access_score = min(memory["access_count"] / 10, 1.0)   # frequency bonus

        # Weighted combination
        relevance = (
            0.4 * semantic_score +
            0.25 * importance_score +
            0.2 * recency_score +
            0.15 * access_score
        )
        return relevance, {
            "semantic": semantic_score,
            "importance": importance_score,
            "recency": recency_score,
            "access": access_score,
        }

    def recall(self, query, k=5, current_time=None):
        """Recall memories with decay-weighted relevance."""
        if self.index.ntotal == 0:
            return []

        query_embedding = embed_texts([query])[0]
        n = min(k * 3, self.index.ntotal)  # fetch more, then re-rank
        scores, indices = self.index.search(np.array([query_embedding], dtype=np.float32), n)

        candidates = []
        for sim, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            memory = self.memories[idx]
            relevance, components = self.compute_relevance(memory, float(sim), current_time)
            candidates.append({
                **memory,
                "similarity": float(sim),
                "relevance": relevance,
                "score_components": components,
                "index": idx,
            })

        # Sort by relevance and return top-k
        candidates.sort(key=lambda x: x["relevance"], reverse=True)

        # Update access counts for returned memories
        for c in candidates[:k]:
            self.memories[c["index"]]["access_count"] += 1
            self.memories[c["index"]]["last_accessed"] = time.time()

        return candidates[:k]

    def forget(self, relevance_threshold=0.1, current_time=None):
        """Remove memories below relevance threshold (garbage collection)."""
        current_time = current_time or time.time()
        to_remove = []

        for i, memory in enumerate(self.memories):
            age_hours = (current_time - memory["created_at"]) / 3600
            recency = math.exp(-self.decay_rate * age_hours)
            importance = memory["importance"] / 10.0
            access = min(memory["access_count"] / 10, 1.0)

            # Retention score (without semantic component)
            retention = 0.4 * importance + 0.35 * recency + 0.25 * access
            if retention < relevance_threshold:
                to_remove.append(i)

        return to_remove  # indices of memories that could be forgotten

    def size(self):
        return len(self.memories)


# Demonstrate decay
dm = DecayingMemory(decay_rate=0.05)  # faster decay for demo

# Store memories with different importance and simulated times
base_time = time.time()

memories_data = [
    ("Critical bug: database connection pool exhaustion under load", 9.0, base_time - 3600 * 1),    # 1 hour ago
    ("User said hello and asked about the weather", 2.0, base_time - 3600 * 2),                     # 2 hours ago
    ("Decision: migrate from REST to GraphQL for the new API", 8.0, base_time - 3600 * 24),         # 1 day ago
    ("Team standup notes: sprint planning next Monday", 4.0, base_time - 3600 * 48),                # 2 days ago
    ("Architecture decision: use event sourcing for audit trail", 9.0, base_time - 3600 * 72),      # 3 days ago
    ("User mentioned they like coffee", 1.0, base_time - 3600 * 1),                                 # 1 hour ago
]

for content, importance, created_at in memories_data:
    dm.store(content, importance=importance)
    dm.memories[-1]["created_at"] = created_at
    dm.memories[-1]["last_accessed"] = created_at

# Some memories have been accessed more
dm.memories[0]["access_count"] = 5  # critical bug — frequently referenced
dm.memories[4]["access_count"] = 3  # architecture decision — referenced sometimes

# Query with decay
print("Memory Recall with Decay Weighting")
print("=" * 80)

results = dm.recall("What are the important technical decisions?", k=5, current_time=base_time)
print(f"\n🔍 Query: 'What are the important technical decisions?'\n")
print(f"{'Relevance':>10} {'Semantic':>9} {'Import':>7} {'Recency':>8} {'Access':>7}  Content")
print("-" * 80)

for r in results:
    c = r["score_components"]
    print(f"  {r['relevance']:.4f}   {c['semantic']:.3f}    {c['importance']:.2f}    {c['recency']:.3f}    {c['access']:.2f}   {r['content'][:45]}")

# Show what would be forgotten
forgettable = dm.forget(relevance_threshold=0.2, current_time=base_time)
print(f"\n🗑️  Memories eligible for forgetting ({len(forgettable)}):")
for idx in forgettable:
    m = dm.memories[idx]
    print(f"   - importance={m['importance']:.0f}, accesses={m['access_count']}: {m['content'][:50]}")

## 8. Comparison: No Memory vs. Short-Term vs. Long-Term

Let's measure the practical difference. We pose the same questions to three agent configurations after a multi-session interaction.

In [ ]:
# Compare memory configurations on cross-session recall
print("Cross-Session Memory Comparison")
print("=" * 70)

# The facts established across sessions
established_facts = [
    "The project is a Python web scraper for e-commerce product prices",
    "It uses BeautifulSoup for HTML parsing",
    "Data is stored in a SQLite database",
    "Email notifications are sent via Gmail SMTP when prices drop",
    "The price threshold is configurable per product",
]

# Test questions that require cross-session knowledge
test_questions = [
    "What parsing library does the scraper use?",
    "How does the notification system work?",
    "What database stores the scraped data?",
]

configs = {
    "No Memory": "You are a helpful assistant.",
    "Short-Term Only": "You are a helpful assistant.",
    "Long-Term Memory": None,  # uses LTM
}

print("\nTest questions:")
for i, q in enumerate(test_questions, 1):
    print(f"  {i}. {q}")
print(f"\n{'─'*70}")

for config_name in configs:
    print(f"\n📌 {config_name}:")

    if config_name == "Long-Term Memory":
        # Use agent with accumulated LTM
        for q in test_questions:
            memories = agent.ltm.recall(q, k=3)
            memory_context = "\n".join([f"- {m.get('content', '')[:80]}" for m in memories])

            messages = [
                {"role": "system", "content": f"You are a helpful assistant.\n\nRelevant memories:\n{memory_context}"},
                {"role": "user", "content": q}
            ]
            response = generate(messages, max_new_tokens=150, temperature=0.3)
            print(f"  Q: {q}")
            print(f"  A: {response[:150]}")
            print()
    elif config_name == "Short-Term Only":
        # Only has current session context (session 3 — no established facts)
        for q in test_questions:
            messages = [
                {"role": "system", "content": configs[config_name]},
                {"role": "user", "content": q}
            ]
            response = generate(messages, max_new_tokens=150, temperature=0.3)
            print(f"  Q: {q}")
            print(f"  A: {response[:150]}")
            print()
    else:
        # No memory at all
        for q in test_questions:
            messages = [
                {"role": "system", "content": configs[config_name]},
                {"role": "user", "content": q}
            ]
            response = generate(messages, max_new_tokens=150, temperature=0.3)
            print(f"  Q: {q}")
            print(f"  A: {response[:150]}")
            print()

print("─" * 70)
print("📊 Only the Long-Term Memory agent can answer from cross-session knowledge.")

## 9. Connection to RAG — LTM IS RAG Over Your Own Experiences

An important insight: **long-term memory is architecturally identical to RAG** (Retrieval-Augmented Generation). The only difference is the data source:

| Component | RAG | Agent LTM |
|-----------|-----|-----------|
| **Data source** | External documents | Agent's own experiences |
| **Embedding** | Document chunks → vectors | Memories → vectors |
| **Storage** | Vector store (FAISS, etc.) | Same vector store |
| **Retrieval** | Query → relevant chunks | Query → relevant memories |
| **Augmentation** | Add chunks to prompt | Add memories to prompt |
| **Generation** | LLM produces answer | LLM produces action/response |

This means everything you learned in the RAG module applies:
- **Chunking strategies** → memory granularity
- **Embedding quality** → recall accuracy
- **Retrieval tuning** (k, threshold) → memory precision/recall
- **Re-ranking** → importance × recency × similarity

In [ ]:
# Demonstrate the RAG-LTM equivalence
print("RAG vs. Agent LTM — Structural Equivalence")
print("=" * 60)

# Both use the same pipeline
pipeline_steps = [
    ("1. Ingest",     "embed(document_chunk)",    "embed(agent_experience)"),
    ("2. Store",      "faiss.add(embedding)",      "faiss.add(embedding)"),
    ("3. Query",      "embed(user_question)",      "embed(current_context)"),
    ("4. Retrieve",   "faiss.search(query, k=5)",  "faiss.search(query, k=5)"),
    ("5. Augment",    "prompt += relevant_chunks",  "prompt += relevant_memories"),
    ("6. Generate",   "llm(augmented_prompt)",      "llm(augmented_prompt)"),
]

print(f"{'Step':<12} {'RAG':<30} {'Agent LTM':<30}")
print("-" * 72)
for step, rag, ltm_step in pipeline_steps:
    print(f"{step:<12} {rag:<30} {ltm_step:<30}")

print("""
✓ The pipeline is identical — only the data source differs.
  RAG: retrieves from external knowledge bases
  LTM: retrieves from the agent's own accumulated experiences

This is why the RAG module is a prerequisite for understanding agent memory.
""")

## 10. Key Takeaways

### What We Built
1. **EpisodicMemory** — FAISS-backed store for timestamped experiences with similarity search
2. **SemanticMemory** — deduplicated fact store with cosine similarity-based dedup
3. **LongTermMemory** — unified class combining episodic + semantic with consolidation
4. **MemoryEnhancedAgent** — ReAct agent that recalls memories and stores new ones
5. **DecayingMemory** — importance × recency × access frequency weighting with forgetting

### Key Insights

- **Long-term memory = RAG over the agent's own experiences** — same architecture, different data source
- **Episodic vs. semantic memory** serve different needs: events vs. facts, with different retrieval patterns
- **Deduplication is essential** — without it, the same fact stored 100 times dominates retrieval
- **Memory decay prevents unbounded growth** — not everything is worth remembering forever
- **Consolidation bridges episodic→semantic** — extracting lasting knowledge from transient experiences
- **Cross-session recall is the killer feature** — transforms an amnesiac chatbot into a knowledgeable assistant

### What's Next
- **Notebook 12**: Knowledge graph memory — structured memory that preserves relationships
- **Notebook 13**: Advanced tool design — building production-grade agent tools
- **Notebook 14**: Code execution tool — agents that write and run code

### References
- Park et al. (2023) — *"Generative Agents: Interactive Simulacra of Human Behavior"*
- Packer et al. (2023) — *"MemGPT: Towards LLMs as Operating Systems"*
- Tulving (1972) — *"Episodic and Semantic Memory"* (original distinction)
- Lewis et al. (2020) — *"Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"*